# Effective mass plot

This notebook combines the selected runs, computes
`m_eff(R,T) = log(W(R,T) / W(R,T+1))`, and saves the final plot as a PDF.

The defaults are set for beta=2.4, eps1=0.0, R=4, thermalization=1000,
bootstrap block size=500, tflow=0, and all 40 matching runs.

In [ ]:
# Edit these values for a different plot.
from pathlib import Path

DATA_ROOT = Path("../data").resolve()
RUN_ROOTS = [DATA_ROOT]
FILTER_TOKENS = ["beta=2.5", "eps1=0.0"]
EXPECTED_RUN_COUNT = 40

R_VALUE = 4
TFLOW = 0.5
PRECOMPUTED_ANALYSIS_DIR = DATA_ROOT / "gradient_flow_wtemp_analysis" / "beta_2p5__L0_32__epsilon1_0__dt_0p01__72d8fe58"
USE_PRECOMPUTED_ANALYSIS = True  # Uses wilson_loop_stats.dat and bootstrap/*.npy instead of raw W_temp files.
USE_W_TEMP_OUT_FOR_TFLOW_ZERO = True  # W_temp.out is the cheaper source for the unflowed/tflow=0 loops.
THERMALIZATION_STEPS = 1000
BOOTSTRAP_BLOCK_SIZE = 500
N_BOOTSTRAP = 1000
LOAD_WORKERS = 4
FORCE_REBUILD_CACHE = False

OUTPUT_DIR = Path("plots").resolve()
CACHE_DIR = Path("notebook_cache").resolve()
PDF_PATH = OUTPUT_DIR / "effective_mass_beta2p5_eps0_R4_tflow0p5_precomputed.pdf"
CACHE_PATH = CACHE_DIR / "effective_mass_beta2p5_eps0_R4_tflow0p5_precomputed.json"

# Plot styling, matching gradient_flow_wtemp_plotting.ipynb PDF exports.
FIGURE_WIDTH = 950
FIGURE_HEIGHT = 600
POINT_COLOR = "#1f77b4"
CAPSIZE = 3
MARKER_SIZE = 7
PDF_MARGIN = {"l": 48, "r": 12, "t": 12, "b": 46}

In [ ]:
import concurrent.futures
import os
import sys
from datetime import datetime, timezone

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import data_organizer as do
import run_evaluation
from calculator import Calculator
from finalized_analysis_helpers import _ensure_plotly_browser_path
from finalized_analysis_helpers import build_effective_mass_scan, build_wrt_scan, load_json, save_json
from finalize_analysis import (
    discover_run_directories_from_roots,
    filter_run_directories,
    parse_filter_tokens,
    wilson_flow_time_filter_label,
    wilson_flow_time_pair_filter,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
_ensure_plotly_browser_path()
pio.templates.default = "plotly_white"

In [ ]:
criteria = parse_filter_tokens(FILTER_TOKENS)
run_dirs = filter_run_directories(
    discover_run_directories_from_roots([str(path) for path in RUN_ROOTS]),
    criteria,
)

if EXPECTED_RUN_COUNT is not None and len(run_dirs) != int(EXPECTED_RUN_COUNT):
    raise RuntimeError(
        f"Expected {EXPECTED_RUN_COUNT} runs, found {len(run_dirs)}. "
        "Edit RUN_ROOTS/FILTER_TOKENS or EXPECTED_RUN_COUNT."
    )

print(f"Selected {len(run_dirs)} runs")
for run_dir in run_dirs:
    print(run_dir)

In [ ]:
def _cache_metadata():
    return {
        "run_dirs": [os.path.abspath(path) for path in run_dirs],
        "filter_tokens": list(FILTER_TOKENS),
        "R": int(R_VALUE),
        "tflow": None if TFLOW is None else float(TFLOW),
        "use_precomputed_analysis": bool(USE_PRECOMPUTED_ANALYSIS),
        "precomputed_analysis_dir": str(PRECOMPUTED_ANALYSIS_DIR.resolve()) if USE_PRECOMPUTED_ANALYSIS else None,
        "use_w_temp_out_for_tflow_zero": bool(USE_W_TEMP_OUT_FOR_TFLOW_ZERO),
        "thermalization_steps": int(THERMALIZATION_STEPS),
        "bootstrap_block_size": int(BOOTSTRAP_BLOCK_SIZE),
        "n_bootstrap": int(N_BOOTSTRAP),
    }


def loader_flow_time():
    if USE_W_TEMP_OUT_FOR_TFLOW_ZERO and TFLOW is not None and np.isclose(float(TFLOW), 0.0):
        return None
    return TFLOW


def selected_wilson_pair_filter(flow_time, r_value, t_value):
    requested_flow_time = loader_flow_time()
    return (
        wilson_flow_time_pair_filter(requested_flow_time)(flow_time, r_value, t_value)
        and int(r_value) == int(R_VALUE)
    )


def load_one_w_temp_out(run_dir, min_step):
    path = Path(run_dir) / "W_temp.out"
    if not path.exists():
        return run_dir, None
    compact = do.load_compact_wilson_file_filtered(
        str(path),
        min_step=int(min_step),
        pair_filter=lambda flow_time, r_value, t_value: flow_time is None and int(r_value) == int(R_VALUE),
    )
    return run_dir, compact


def load_combined_w_temp_for_effective_mass(cuts_by_run):
    requested_flow_time = loader_flow_time()
    if requested_flow_time is not None:
        return run_evaluation._load_combined_w_temp_filtered(
            run_dirs,
            pair_filter=selected_wilson_pair_filter,
            filter_label=f"{wilson_flow_time_filter_label(requested_flow_time)}, R={int(R_VALUE)}",
            verbose=True,
            prefix="[effective_mass_notebook]",
            load_workers=int(LOAD_WORKERS),
            thermalization_steps_by_run=cuts_by_run,
        )

    worker_count = max(1, min(int(LOAD_WORKERS), len(run_dirs)))
    print(f"[effective_mass_notebook] Loading only W_temp.out with up to {worker_count} worker(s).")

    def load_current(run_dir):
        return load_one_w_temp_out(run_dir, cuts_by_run[os.path.abspath(run_dir)])

    if worker_count > 1:
        with concurrent.futures.ThreadPoolExecutor(max_workers=worker_count) as executor:
            loaded = list(executor.map(load_current, run_dirs))
    else:
        loaded = [load_current(run_dir) for run_dir in run_dirs]

    compact_parts = [compact for _run_dir, compact in loaded if compact is not None]
    combined = do.combine_compact_wilson_data(compact_parts, source_name="W_temp_R_filtered_combined")
    sample_counts = getattr(combined, "pair_sample_counts", {}) if combined is not None else {}
    sample_lengths = [int(value) for value in sample_counts.values()]
    aggregation = {
        "n_runs_in_group": len(run_dirs),
        "n_runs_with_w_temp": len(compact_parts),
        "n_w_temp_files": len(compact_parts),
        "n_samples_after_cut": int(sum(sample_lengths)),
        "n_configurations_after_cut": int(max(sample_lengths, default=0)),
        "thermalization_steps": int(THERMALIZATION_STEPS),
        "thermalization_steps_by_run": cuts_by_run,
        "load_workers": worker_count,
        "wilson_loop_filter": f"W_temp.out only, R={int(R_VALUE)}",
    }
    return combined, aggregation


def read_precomputed_wilson_stats(analysis_dir, *, flow_time, r_value):
    stats_path = Path(analysis_dir) / "wilson_loop_stats.dat"
    if not stats_path.exists():
        raise FileNotFoundError(f"Missing precomputed stats file: {stats_path}")

    rows = []
    with stats_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            if len(parts) < 12:
                continue
            row = {
                "t_over_a2": float(parts[0]),
                "R": int(parts[1]),
                "T": int(parts[2]),
                "n_measurements": int(parts[3]),
                "first_conf_id": int(float(parts[4])),
                "last_conf_id": int(float(parts[5])),
                "block_size": int(parts[6]),
                "n_blocks": int(parts[7]),
                "tail": int(parts[8]),
                "value": float(parts[9]),
                "err": float(parts[10]),
                "bootstrap_path": str(Path(analysis_dir) / parts[11]),
            }
            if np.isclose(row["t_over_a2"], float(flow_time)) and row["R"] == int(r_value):
                rows.append(row)
    rows = sorted(rows, key=lambda row: row["T"])
    if len(rows) < 2:
        raise RuntimeError(f"Need at least two precomputed T values for R={r_value}, tflow={flow_time}.")
    return rows


def build_effective_mass_from_precomputed_analysis():
    if TFLOW is None:
        raise ValueError("Precomputed gradient-flow analysis requires a numeric TFLOW.")
    wrt_records = read_precomputed_wilson_stats(PRECOMPUTED_ANALYSIS_DIR, flow_time=float(TFLOW), r_value=int(R_VALUE))
    by_t = {int(row["T"]): row for row in wrt_records}
    effective_mass_records = []
    for t_value in sorted(by_t)[:-1]:
        current = by_t[t_value]
        following = by_t.get(t_value + 1)
        if following is None:
            continue
        if current["value"] <= 0 or following["value"] <= 0:
            continue
        value = float(np.log(current["value"] / following["value"]))
        current_boot = np.load(current["bootstrap_path"])
        following_boot = np.load(following["bootstrap_path"])
        valid = np.isfinite(current_boot) & np.isfinite(following_boot) & (current_boot > 0) & (following_boot > 0)
        boot = np.full(np.asarray(current_boot).shape, np.nan, dtype=float)
        boot[valid] = np.log(current_boot[valid] / following_boot[valid])
        finite_boot = boot[np.isfinite(boot)]
        effective_mass_records.append(
            {
                "R": int(R_VALUE),
                "T": int(t_value),
                "t_mid": float(t_value) + 0.5,
                "value": value,
                "err": float(np.std(finite_boot)) if finite_boot.size else None,
                "flow_time": float(TFLOW),
                "bootstrap_finite_fraction": float(finite_boot.size / boot.size) if boot.size else None,
            }
        )

    block_sizes = sorted({int(row["block_size"]) for row in wrt_records})
    n_bootstraps = sorted({int(np.load(row["bootstrap_path"]).size) for row in wrt_records})
    return {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "metadata": _cache_metadata(),
        "aggregation": {
            "source": "precomputed_gradient_flow_wtemp_analysis",
            "analysis_dir": str(PRECOMPUTED_ANALYSIS_DIR.resolve()),
            "n_runs_in_group": len(run_dirs),
            "n_runs_with_w_temp": len(run_dirs),
            "n_w_temp_files": len(run_dirs),
            "thermalization_steps": int(THERMALIZATION_STEPS),
            "wilson_loop_filter": f"precomputed t_over_a2={float(TFLOW):g}, R={int(R_VALUE)}",
            "precomputed_block_sizes": block_sizes,
            "precomputed_n_bootstrap": n_bootstraps,
        },
        "loader_flow_time": float(TFLOW),
        "available_flow_times": [float(TFLOW)],
        "available_T": [int(row["T"]) for row in wrt_records],
        "wrt_records": wrt_records,
        "effective_mass_records": effective_mass_records,
    }


def build_effective_mass_payload():
    if USE_PRECOMPUTED_ANALYSIS:
        return build_effective_mass_from_precomputed_analysis()

    cuts_by_run = {
        os.path.abspath(run_dir): int(THERMALIZATION_STEPS)
        for run_dir in run_dirs
    }
    requested_flow_time = loader_flow_time()
    combined_w_temp, aggregation = load_combined_w_temp_for_effective_mass(cuts_by_run)
    if combined_w_temp is None:
        raise RuntimeError("No combined W_temp data could be loaded from the selected runs.")

    calc = Calculator(
        combined_w_temp,
        n_bootstrap=int(N_BOOTSTRAP),
        step_size=max(1, int(BOOTSTRAP_BLOCK_SIZE)),
    )
    available_flow_times = calc.get_available_flow_times()
    unique_ts = [int(t) for t in calc.get_unique_Ts(R=int(R_VALUE), flow_time=requested_flow_time)]
    if len(unique_ts) < 2:
        raise RuntimeError(f"Need at least two T values for R={R_VALUE}; got {unique_ts}.")

    calc.prime_w_rt_cache([(int(R_VALUE), t_value) for t_value in unique_ts], flow_time=requested_flow_time)
    wrt_records = build_wrt_scan(calc, [int(R_VALUE)], unique_ts, flow_time=requested_flow_time)
    effective_mass_records = build_effective_mass_scan(calc, [int(R_VALUE)], unique_ts, flow_time=requested_flow_time)

    return {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "metadata": _cache_metadata(),
        "aggregation": aggregation,
        "loader_flow_time": None if requested_flow_time is None else float(requested_flow_time),
        "available_flow_times": [None if value is None else float(value) for value in available_flow_times],
        "available_T": unique_ts,
        "wrt_records": wrt_records,
        "effective_mass_records": effective_mass_records,
    }


payload = load_json(CACHE_PATH, default=None)
if FORCE_REBUILD_CACHE or payload is None or payload.get("metadata") != _cache_metadata():
    payload = build_effective_mass_payload()
    save_json(CACHE_PATH, payload)
    print(f"Saved cache: {CACHE_PATH}")
else:
    print(f"Loaded cache: {CACHE_PATH}")

print("Aggregation:", payload.get("aggregation", {}))

In [ ]:
eff_records = sorted(payload["effective_mass_records"], key=lambda row: int(row["T"]))
wrt_records = sorted(payload["wrt_records"], key=lambda row: int(row["T"]))

if not eff_records:
    raise RuntimeError("No finite effective-mass records were produced.")

print("T  t_mid  m_eff  err")
for row in eff_records:
    err_text = "nan" if row.get("err") is None else f"{float(row['err']):.8g}"
    print(f"{int(row['T']):2d} {float(row['t_mid']):5.1f} {float(row['value']):.8g} {err_text}")
print(f"Effective-mass points: {len(eff_records)}")

In [ ]:
def make_effective_mass_pdf_figure(eff_records):
    mass_x = np.asarray([float(row["t_mid"]) for row in eff_records], dtype=float)
    mass_y = np.asarray([float(row["value"]) for row in eff_records], dtype=float)
    mass_yerr = np.asarray([0.0 if row.get("err") is None else float(row["err"]) for row in eff_records], dtype=float)
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=mass_x,
            y=mass_y,
            mode="markers",
            name=f"R={R_VALUE}",
            marker={"size": MARKER_SIZE, "color": POINT_COLOR},
            error_y={"type": "data", "array": mass_yerr, "visible": True},
            customdata=np.column_stack([
                [int(row["T"]) for row in eff_records],
                mass_yerr,
            ]),
            hovertemplate=(
                "T=%{customdata[0]}<br>"
                "T+1/2=%{x:g}<br>"
                "m_eff=%{y:.8g}<br>"
                "bootstrap error=%{customdata[1]:.3g}"
                "<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        title=None,
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT,
        xaxis_title="T + 1/2",
        yaxis_title="m_eff(R,T)",
        margin=PDF_MARGIN,
        showlegend=False,
    )
    return fig


fig = make_effective_mass_pdf_figure(eff_records)
fig.write_image(str(PDF_PATH))
print(f"Saved PDF: {PDF_PATH}")
fig